# Notebook 3: LAPT and Vocabulary Augmentation

**Goal**: Reproduce the LAPT, VA, and TVA results from the paper.

This is the core of the paper. We:
1. Run **LAPT** — continue pretraining mBERT on each language's unlabeled text
2. Run **VA** — augment mBERT vocabulary with language-specific wordpieces, then LAPT
3. Train the parser on top of each adapted model

**Expected improvement over baseline (MBERT+FT):**

| Method | Irish | Maltese | Singlish | Vietnamese |
|---|---|---|---|---|
| MBERT+FT | 72.67 | 76.74 | 78.24 | 66.13 |
| LAPT+FT | **75.45** | **82.77** | **79.30** | **67.50** |
| VA+FT | **76.17** | **83.53** | **79.89** | **67.28** |

**Note:** LAPT takes 1-4 hours per language depending on dataset size.
Use Colab Pro or run one language at a time.

In [ ]:
# Environment setup
import os, sys
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR  = '/content/parsing-mbert'
WORKSPACE = '/content/drive/MyDrive/thesis_mbert'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone https://github.com/Sansgithub22/mtechthesis2Parsing-mbert.git {REPO_DIR}')

sys.path.insert(0, os.path.join(REPO_DIR, 'thesis', 'src'))

# Colab already has PyTorch — only install the missing packages
!pip install -q transformers==4.38.0 tokenizers==0.15.2 datasets==2.18.0 peft==0.10.0 accelerate==0.27.0 conllu==4.5.3 sentencepiece==0.1.99 tqdm seaborn

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch: {torch.__version__} | Device: {DEVICE}')

In [ ]:
# Data paths
UD_BASE = os.path.join(WORKSPACE, 'ud-treebanks-v2.5')

TREEBANKS = {
    'ga':   {'train': f'{UD_BASE}/UD_Irish-IDT/ga_idt-ud-train.conllu',
             'dev':   f'{UD_BASE}/UD_Irish-IDT/ga_idt-ud-dev.conllu',
             'test':  f'{UD_BASE}/UD_Irish-IDT/ga_idt-ud-test.conllu'},
    'mt':   {'train': f'{UD_BASE}/UD_Maltese-MUDT/mt_mudt-ud-train.conllu',
             'dev':   f'{UD_BASE}/UD_Maltese-MUDT/mt_mudt-ud-dev.conllu',
             'test':  f'{UD_BASE}/UD_Maltese-MUDT/mt_mudt-ud-test.conllu'},
    'vi':   {'train': f'{UD_BASE}/UD_Vietnamese-VTB/vi_vtb-ud-train.conllu',
             'dev':   f'{UD_BASE}/UD_Vietnamese-VTB/vi_vtb-ud-dev.conllu',
             'test':  f'{UD_BASE}/UD_Vietnamese-VTB/vi_vtb-ud-test.conllu'},
    'sing': {'train': f'{REPO_DIR}/data/sing/train.conllu',
             'dev':   f'{REPO_DIR}/data/sing/dev.conllu',
             'test':  f'{REPO_DIR}/data/sing/test.conllu'},
}

UNLABELED = {
    'ga':   f'{REPO_DIR}/data/unlabeled/ga',
    'mt':   f'{REPO_DIR}/data/unlabeled/mt',
    'vi':   f'{REPO_DIR}/data/unlabeled/vi',
    'sing': f'{REPO_DIR}/data/unlabeled/sg',
}

# Epochs from the paper (Table 5)
LAPT_EPOCHS = {'ga': 5, 'mt': 20, 'vi': 5, 'sing': 5}
VA_EPOCHS   = {'ga': 10, 'mt': 15, 'vi': 5, 'sing': 1}

print('Paths configured.')

In [ ]:
# ============================================================
# PART 1: LANGUAGE-ADAPTIVE PRETRAINING (LAPT)
# ============================================================
# Continue pretraining mBERT on each language's unlabeled text.
# This adapts the wordpiece embeddings to the target language.

from training.lapt import run_lapt

# CHANGE THIS to run a specific language
# Run one at a time to avoid Colab timeouts
LANG = 'mt'   # 'ga', 'mt', 'vi', 'sing'

lapt_model_dir = os.path.join(WORKSPACE, 'lapt_models', f'lapt_{LANG}')

if os.path.exists(os.path.join(lapt_model_dir, 'config.json')):
    print(f'LAPT model for {LANG} already exists at {lapt_model_dir}')
else:
    print(f'Running LAPT for {LANG.upper()} with {LAPT_EPOCHS[LANG]} epochs...')
    run_lapt(
        model_name_or_path='bert-base-multilingual-cased',
        train_text_path=os.path.join(UNLABELED[LANG], 'train.txt'),
        eval_text_path=os.path.join(UNLABELED[LANG], 'valid.txt'),
        output_dir=lapt_model_dir,
        num_epochs=LAPT_EPOCHS[LANG],
        batch_size=16,
        learning_rate=2e-5,
        warmup_steps=1000,
        fp16=True,
    )

print(f'LAPT model for {LANG}: {lapt_model_dir}')

In [ ]:
# Train the parser on top of the LAPT-adapted model
# (reuse run_experiment from notebook 02)

import json
from transformers import AutoTokenizer
from data.conllu_reader import read_conllu, get_relation_vocab
from models.encoder import BERTEncoder
from models.biaffine_parser import BiaffineParser
from training.trainer import ParserTrainer

def run_experiment(lang, model_path, freeze_bert, experiment_name,
                   max_epochs=200, patience=20, batch_size=24,
                   bert_lr=5e-5, parser_lr=1e-3):
    print(f'\n{"="*60}')
    print(f'Experiment: {experiment_name} | Language: {lang.upper()}')
    print(f'{"="*60}\n')

    paths = TREEBANKS[lang]
    train_sents = read_conllu(paths['train'])
    dev_sents   = read_conllu(paths['dev'])
    test_sents  = read_conllu(paths['test'])

    rel_vocab = get_relation_vocab(train_sents)
    n_rels = max(rel_vocab.values()) + 1

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    encoder = BERTEncoder(model_name=model_path, freeze=freeze_bert, dropout=0.1)

    model = BiaffineParser(
        encoder=encoder, n_rels=n_rels,
        arc_dim=500, rel_dim=100,
        bilstm_layers=3, bilstm_hidden=400,
        mlp_dropout=0.33,
        use_bilstm=freeze_bert,
    )

    save_dir = os.path.join(WORKSPACE, 'results', experiment_name, lang)
    os.makedirs(save_dir, exist_ok=True)

    trainer = ParserTrainer(
        model=model, train_sentences=train_sents, dev_sentences=dev_sents,
        tokenizer=tokenizer, rel_vocab=rel_vocab, save_dir=save_dir,
        batch_size=batch_size, max_epochs=max_epochs, patience=patience,
        bert_lr=bert_lr, parser_lr=parser_lr, device=DEVICE,
    )

    best_dev_las = trainer.train()
    test_las, test_uas = trainer.evaluate_test(test_sents, tokenizer)

    result = {'lang': lang, 'experiment': experiment_name,
              'best_dev_las': round(best_dev_las, 2),
              'test_las': round(test_las, 2), 'test_uas': round(test_uas, 2)}

    with open(os.path.join(save_dir, 'result.json'), 'w') as f:
        json.dump(result, f, indent=2)
    return result


# Train parser with LAPT model (FT variant)
lapt_ft_result = run_experiment(
    lang=LANG,
    model_path=lapt_model_dir,
    freeze_bert=False,
    experiment_name='lapt_ft',
)
print(f'LAPT+FT [{LANG}] LAS = {lapt_ft_result["test_las"]}')

# Compare with paper
PAPER_LAPT_FT = {'ga': 75.45, 'mt': 82.77, 'vi': 67.50, 'sing': 79.30}
diff = lapt_ft_result['test_las'] - PAPER_LAPT_FT.get(LANG, 0)
print(f'Paper LAPT+FT [{LANG}] = {PAPER_LAPT_FT.get(LANG)}  |  Diff = {diff:+.2f}')

In [ ]:
# ============================================================
# PART 2: VOCABULARY AUGMENTATION (VA)
# ============================================================
# 1. Train a new wordpiece vocab on target language data
# 2. Inject novel wordpieces into mBERT
# 3. Run LAPT on the augmented model

from vocab.augmentation import run_va_pipeline

va_output_dir = os.path.join(WORKSPACE, 'va_models', f'va_{LANG}')
va_model_dir  = os.path.join(va_output_dir, 'va_lapt')

if os.path.exists(os.path.join(va_model_dir, 'config.json')):
    print(f'VA model for {LANG} already exists at {va_model_dir}')
else:
    print(f'Running VA pipeline for {LANG.upper()}...')
    va_model_dir = run_va_pipeline(
        base_model='bert-base-multilingual-cased',
        text_path=os.path.join(UNLABELED[LANG], 'train.txt'),
        eval_path=os.path.join(UNLABELED[LANG], 'valid.txt'),
        output_dir=va_output_dir,
        vocab_size=5000,
        top_k=99,
        num_lapt_epochs=VA_EPOCHS[LANG],
        batch_size=16,
        learning_rate=2e-5,
        fp16=True,
    )

print(f'VA model: {va_model_dir}')

In [ ]:
# Train parser on top of VA model (FT variant)

va_ft_result = run_experiment(
    lang=LANG,
    model_path=va_model_dir,
    freeze_bert=False,
    experiment_name='va_ft',
)
print(f'VA+FT [{LANG}] LAS = {va_ft_result["test_las"]}')

PAPER_VA_FT = {'ga': 76.17, 'mt': 83.53, 'vi': 67.28, 'sing': 79.89}
diff = va_ft_result['test_las'] - PAPER_VA_FT.get(LANG, 0)
print(f'Paper VA+FT [{LANG}] = {PAPER_VA_FT.get(LANG)}  |  Diff = {diff:+.2f}')

In [ ]:
# Summary: Compare all results for the current language
print(f'\nResults summary for {LANG.upper()}')
print(f'{"Method":<20} {"Our LAS":>10} {"Paper LAS":>10}')
print('-' * 42)

comparisons = [
    ('MBERT frozen', 'mbert_frozen', {'ga':68.19,'mt':67.06,'vi':62.96,'sing':74.01}),
    ('MBERT+FT',     'mbert_ft',     {'ga':72.67,'mt':76.74,'vi':66.13,'sing':78.24}),
    ('LAPT+FT',      'lapt_ft',      {'ga':75.45,'mt':82.77,'vi':67.50,'sing':79.30}),
    ('VA+FT',        'va_ft',        {'ga':76.17,'mt':83.53,'vi':67.28,'sing':79.89}),
]

for method_name, exp_name, paper_vals in comparisons:
    result_path = os.path.join(WORKSPACE, 'results', exp_name, LANG, 'result.json')
    if os.path.exists(result_path):
        with open(result_path) as f:
            r = json.load(f)
        our = r['test_las']
    else:
        our = 'N/A'
    paper = paper_vals.get(LANG, 'N/A')
    print(f'{method_name:<20} {str(our):>10} {str(paper):>10}')